# Amazon ML Challenge 2026 — Google Colab Runtime Verification & Experiment Runner
## Business Entity Resolution

**Objective**: Verified, reproducible environment verification and candidate generation strategy benchmarking on Google Colab (GPU/CPU).

### Strict Operating Rules:
1. **No Datasets in GitHub**: Datasets are stored outside Git and mounted via Google Drive (`/content/drive/MyDrive/DATASET_ML-AMAZON/dataset`).
2. **Strict Verification**: Fails immediately if GPU is missing, Drive path is invalid, or any required TSV file is missing.
3. **Zero RAM Exhaustion**: Datasets are inspected via streaming chunked reads without loading full files into memory.
4. **No Premature Full Pipeline**: Validates environment, path resolution, and candidate blocking metrics without unconstrained all-pairs cartesian products.

### Step 1: Verify Google Colab Runtime

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f"Running inside Google Colab: {IN_COLAB}")
if not IN_COLAB:
    print("[NOTICE] If running locally, please ensure paths and hardware settings match your local environment.")

### Step 2: GPU & PyTorch CUDA Availability Check
Ensures an active CUDA GPU is assigned (e.g., T4, V100, A100).

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "[ERROR] CUDA GPU is unavailable! Please switch runtime in Colab menu: "
    "Runtime -> Change runtime type -> Hardware accelerator: T4 GPU or A100 GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
cuda_ver = torch.version.cuda
torch_ver = torch.__version__

print(f"[OK] CUDA Available:     True")
print(f"[OK] GPU Device:         {gpu_name}")
print(f"[OK] Total VRAM:         {vram_gb} GB")
print(f"[OK] PyTorch Version:    {torch_ver}")
print(f"[OK] CUDA Version:       {cuda_ver}")

### Step 3: Clone / Sync GitHub Repository & Verify Code Files
Synchronizes latest main commit from `https://github.com/Toxic012/ML_AMAZON-HACKTHON.git` and verifies all runner and blocking scripts exist.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Toxic012/ML_AMAZON-HACKTHON.git"
REPO_NAME = "ML_AMAZON-HACKTHON"

if 'google.colab' in sys.modules:
    REPO_DIR = Path("/content") / REPO_NAME
    if not (REPO_DIR / ".git").exists():
        print(f"Cloning fresh repository into {REPO_DIR}...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f"Updating existing repository at {REPO_DIR}...")
        !git -C {REPO_DIR} fetch origin
        !git -C {REPO_DIR} checkout main
        !git -C {REPO_DIR} reset --hard origin/main
        !git -C {REPO_DIR} pull origin main
    %cd {REPO_DIR}
else:
    REPO_DIR = Path(".").resolve()
    !git fetch origin
    !git checkout main
    !git pull origin main

# Ensure repo is in sys.path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True, cwd=str(REPO_DIR)).strip()
git_status = subprocess.check_output(["git", "status", "--short"], text=True, cwd=str(REPO_DIR)).strip()

print("=" * 70)
print(f"[OK] Active Working Directory: {Path.cwd()}")
print(f"[OK] Checked-out Commit SHA:   {current_commit}")
print(f"[OK] Git Status:               {git_status or 'Clean'}")
print("=" * 70)

# Explicit verification of all required source files
REQUIRED_CODE_FILES = [
    "scripts/colab_runner.py",
    "scripts/run_blocking_eval.py",
    "code/business_entity_resolution/src/config.py",
    "code/business_entity_resolution/src/data_loading.py",
    "code/business_entity_resolution/src/normalization.py",
    "code/business_entity_resolution/src/blocking/token_index.py",
    "code/business_entity_resolution/src/blocking/strategies.py",
    "code/business_entity_resolution/src/blocking/evaluator.py",
]

print("Verifying required codebase files:")
for rel_f in REQUIRED_CODE_FILES:
    p = Path(rel_f)
    assert p.exists(), f"[FATAL ERROR] Required file missing: {p.resolve()}"
    print(f"  [OK] Found: {rel_f}")

print("\n[OK] All codebase files verified successfully!")

### Step 4: Install Essential Dependencies

In [ ]:
!pip install -q polars lightgbm rapidfuzz sentence-transformers faiss-cpu psutil
print("[OK] Required packages installed.")

### Step 5: Mount Google Drive & Locate Dataset
Locates the confirmed dataset directory at `/content/drive/MyDrive/DATASET_ML-AMAZON/dataset`.

In [ ]:
import os
from pathlib import Path

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[OK] Google Drive mounted.")
except Exception as e:
    print(f"[INFO] Drive mount note: {e}")

# Check configured or candidate paths
custom_env_path = os.environ.get("AMAZON_ML_DATA_DIR")
candidates = []
if custom_env_path:
    candidates.append(Path(custom_env_path))

candidates.extend([
    Path("/content/drive/MyDrive/DATASET_ML-AMAZON/dataset"),
    Path("/content/dataset"),
    Path("/content/student_resource/dataset"),
    Path("/content/drive/MyDrive/ML_AMAZON/student_resource/dataset"),
    Path("student_resource/dataset").resolve()
])

DATA_DIR = None
for cand in candidates:
    if cand.exists() and (cand / "train" / "train_source1.tsv").exists():
        DATA_DIR = cand
        break

if DATA_DIR is None:
    candidate_str = "\n  - ".join(str(c) for c in candidates)
    raise FileNotFoundError(
        f"[ERROR] DATASET_PATH_NOT_FOUND! Looked in candidate locations:\n  - {candidate_str}\n\n"
        f"Please verify that Google Drive is mounted and the folder\n"
        f"  'My Drive/DATASET_ML-AMAZON/dataset'\n"
        f"exists and contains the train/ and test/ folders."
    )

print(f"[OK] DATASET_PATH_FOUND: {DATA_DIR}")
os.environ["AMAZON_ML_DATA_DIR"] = str(DATA_DIR)

### Step 6: Stream-Audit All 7 TSV Dataset Files (Non-RAM Exhaustive)

In [ ]:
REQUIRED_FILES = [
    "train/train_source1.tsv",
    "train/train_source2.tsv",
    "train/train_source3.tsv",
    "train/train_ground_truth.tsv",
    "test/test_source1.tsv",
    "test/test_source2.tsv",
    "test/test_source3.tsv"
]

print(f"Verifying all 7 dataset files in {DATA_DIR}...")
total_size_mb = 0.0

for rel_path in REQUIRED_FILES:
    fpath = DATA_DIR / rel_path
    assert fpath.exists(), f"[ERROR] Required file missing: {fpath}"
    size_mb = round(fpath.stat().st_size / (1024 ** 2), 2)
    total_size_mb += size_mb
    
    # Read header and line count via stream chunking
    with open(fpath, "r", encoding="utf-8") as f:
        header = f.readline().strip().split("\t")
        
    lines = 0
    with open(fpath, "rb") as f:
        while chunk := f.read(1024 * 1024):
            lines += chunk.count(b"\n")
            
    print(f"  [OK] {rel_path:30s} | {size_mb:8.2f} MB | {max(0, lines - 1):>9,} rows | cols: {len(header)}")

print(f"\n[OK] All 7 dataset files verified successfully! Total Size: {total_size_mb:.2f} MB")

### Step 7: Run Full Colab Environment Check (EXP-0001)
Runs [scripts/colab_runner.py](file:///c:/Users/HP/Desktop/ML_AMAZON/scripts/colab_runner.py) to audit the full CPU 5-stage pipeline, GPU inference, and save telemetry to `experiments/EXP-0001_colab_env_check/`.

In [ ]:
assert Path("scripts/colab_runner.py").exists(), "[FATAL] scripts/colab_runner.py not found! Run Step 3 first."
!python scripts/colab_runner.py --mode check --data-dir "{DATA_DIR}"

### Step 8: Standalone CUDA Tensor Microbenchmark
Executes a direct GPU matrix multiplication and synchronization test.

In [ ]:
import time
import torch

device = torch.device("cuda:0")
print(f"Running CUDA microbenchmark on {torch.cuda.get_device_name(0)}...")

# Warmup
_ = torch.matmul(torch.randn(500, 500, device=device), torch.randn(500, 500, device=device))
torch.cuda.synchronize()

# Benchmark 4000x4000 float32 matmul
t0 = time.perf_counter()
A = torch.randn(4000, 4000, device=device, dtype=torch.float32)
B = torch.randn(4000, 4000, device=device, dtype=torch.float32)
C = torch.matmul(A, B)
torch.cuda.synchronize()
t1 = time.perf_counter()

latency_ms = round((t1 - t0) * 1000, 2)
assert not torch.isnan(C).any(), "[ERROR] CUDA matmul produced NaN values!"
print(f"[OK] 4000x4000 Matrix Multiplication: {latency_ms} ms (Success: True)")

### Step 9: EXP-0002 — Candidate Generation / Blocking Strategy Evaluation
Compares candidate generation strategies against ground truth on validation slice (e.g. 5,000 S1 queries against indexed S2/S3 targets):
- **Strategy 1 (`exact_name`)**: Exact normalized business name matching.
- **Strategy 2 (`rare_tokens`)**: Inverted index matching on non-stopword tokens with frequency capping.
- **Strategy 3 (`disjunctive_union`)**: Exact matches + High-IDF token overlap.
- **Strategy 4 (`composite_union`)**: Exact matches + Multi-token overlap + Address tokens.

Measures True-Pair Recall, S2/S3 breakdown, candidate explosion metrics (mean, p95, max candidates/S1), QPS, and memory.

In [ ]:
from pathlib import Path

# Pre-execution assertions
script_path = Path("scripts/run_blocking_eval.py")
assert script_path.exists(), (
    f"[FATAL ERROR] {script_path} not found in {Path.cwd()}!\n"
    "Please ensure you ran Step 3 to clone/pull the latest repository commit."
)
assert 'DATA_DIR' in locals() and DATA_DIR is not None, "[FATAL ERROR] DATA_DIR is not defined! Run Step 5 first."

print(f"Executing {script_path} with dataset at: {DATA_DIR}")
# Run blocking evaluation on 5,000 queries against 200,000 target records per source
!python scripts/run_blocking_eval.py --sample-size 5000 --target-sample-size 200000 --top-k 50 --experiment-id EXP-0002_blocking_eval --data-dir "{DATA_DIR}"

### Step 10: Display Blocking Experiment Telemetry (EXP-0002)

In [ ]:
import json
from pathlib import Path

report_path = Path("experiments/EXP-0002_blocking_eval/results.json")
if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    print("=" * 75)
    print("EXP-0002 BLOCKING STRATEGY COMPARISON SUMMARY:")
    print("=" * 75)
    print(f"Experiment ID: {results.get('experiment_id')}")
    print(f"Git Commit:    {results.get('git_commit')}")
    print(f"S1 Evaluated:  {results.get('configuration', {}).get('s1_sample_size')}")
    print(f"Target S2/S3:  {results.get('configuration', {}).get('target_sample_size')}")
    print("-" * 75)
    for strat, m in results.get("strategy_results", {}).items():
        print(f"Strategy: {strat:<20}")
        print(f"  - True-Pair Recall:     {m.get('true_pair_recall')*100:.2f}%")
        print(f"  - S2 Recall:            {m.get('s2_recall')*100:.2f}%")
        print(f"  - S3 Recall:            {m.get('s3_recall')*100:.2f}%")
        print(f"  - S1 Entity Coverage:   {m.get('s1_entity_coverage')*100:.2f}%")
        print(f"  - Mean Candidates/S1:   {m.get('candidate_volume', {}).get('mean_per_s1')}")
        print(f"  - P95 Candidates/S1:    {m.get('candidate_volume', {}).get('p95_per_s1')}")
        print(f"  - Queries Per Sec (QPS):{m.get('performance', {}).get('queries_per_sec')}")
        print("-" * 75)
else:
    print("No EXP-0002 report found. Run Step 9 first.")